# Pandas: Basics

*Pandas is a Python library for creating and manipulating DataFrames — two-dimensional objects designed to store data.*

Key capabilities:
- High-performance manipulation of text, integers, numbers, and dates
- Data alignment, reshaping, and pivoting
- Intelligent slicing, grouping, and subsetting
- Merging and joining datasets

Resources: [Official Pandas Docs](https://pandas.pydata.org/about/) · [W3Schools Pandas](https://www.w3schools.com/python/pandas/default.asp) · [Pandas Cheat Sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf)

| | Contents |
|-|----------|
| 1. | [Introduction to DataFrames](#1-introduction-to-dataframes) |
| 2. | [Working with Rows](#2-working-with-rows) |
| 3. | [Working with Columns](#3-working-with-columns) |
| 4. | [Sort and Count](#4-sort-and-count) |
| 5. | [Combining DataFrames](#5-combining-dataframes) |
| 6. | [Making the Most of Pandas](#6-making-the-most-of-pandas) |

## 1. Introduction to DataFrames

Pandas **DataFrames** are the basic unit upon which all operations take place. Think of them as spreadsheets: rows of records and named columns.

- Pandas can import data from many sources — **CSV** files, **Excel** spreadsheets, and also **JSON** (JavaScript Object Notation), a format that looks a lot like a Python list of dictionaries. These can be loaded locally (from your computer or Jupyter Hub space) or remotely (from a URL).

- Real-world JSON often contains **nested objects** (a value that is itself a dict) or **arrays** (a value that is a list). We need to **normalize** that structure into a flat table before we can work with it in Pandas.



### Import Libraries

`requests` fetches data from a URL; `pandas` turns it into a DataFrame.

In [1]:
import requests
import pandas as pd

### Meet the CRIM Dataset

We will work with data from the **CRIM Project** (Citations: The Renaissance Imitation Mass) — a scholarly initiative that encodes and analyses polyphonic music from the 16th century. The dataset we will load contains metadata about the **Model compositions** in the project: the pre-existing pieces (motets, chansons, madrigals, plainchant, etc.) that later composers cited, imitated, or transformed when writing Mass settings

The endpoint we are loading contains **metadata about the Model compositions** in the project: the pre-existing pieces (motets, chansons, madrigals, plainchant, etc.) that later composers cited, imitated, or transformed when writing Mass settings. Each record describes one model work and includes:

- Composer and title
- Genre and date
- Number of voices
- Links to MEI (digital score) and PDF files

Each record in the JSON has this structure:

```json
{
  "piece_id":         "CRIM_Model_0001",
  "title":            "Vidi speciosam",
  "full_title":       "Vidi speciosam (motet à 5)",
  "genre":            { "url": "...", "name": "motet" },
  "composer":         { "url": "...", "name": "Lupi, Johannes" },
  "date":             "1538",
  "date_sort":        1538,
  "number_of_voices": 5,
  "remarks":          "...",
  "mei_links":        ["https://..."],
  "pdf_links":        ["https://..."]
}
```

The `genre` and `composer` fields are **nested objects**.  That is, they are dictionaries within the JSON structure   — we need to flatten them into columns before we can work with the data in Pandas.

### Load and Normalize the JSON

`pd.json_normalize()` flattens nested dicts automatically, turning `genre.name` and `composer.name` into their own columns. We then clean up column names and drop the URL columns we don't need.

In [2]:
url = 'https://crimproject.org/data/models/'
response = requests.get(url)
raw_data = response.json()   # a Python list of dicts

# Flatten nested genre/composer objects
crim = pd.json_normalize(raw_data)

# Rename dotted columns to cleaner names
crim = crim.rename(columns={
    'genre.name':    'genre',
    'composer.name': 'composer'
})

# Drop columns we don't need for this tutorial
crim = crim.drop(columns=['url', 'genre.url', 'composer.url'], errors='ignore')

crim.head(5)

,piece_id,title,full_title,pdf_links,mei_links,date,date_sort,number_of_voices,remarks,genre,composer
0,CRIM_Model_0001,Vidi speciosam,Vidi speciosam,[https://crimproject.org/pdf/CRIM_Model_0001.pdf],[https://crimproject.org/mei/CRIM_Model_0001.mei],1538,1538.0,5,,Motet,Johannes Lupi
1,CRIM_Model_0002,O gente brunette,O gente brunette,[https://crimproject.org/pdf/CRIM_Model_0002.pdf],[https://crimproject.org/mei/CRIM_Model_0002.mei],1548,1548.0,4,,Chanson,Thomas Champion
2,CRIM_Model_0003,Missa IX (Cum iubilo) - Kyrie,Missa IX (Cum iubilo) - Kyrie,[https://crimproject.org/pdf/CRIM_Model_0003.pdf],[https://crimproject.org/mei/CRIM_Model_0003.mei],11xx,1100.0,1,In solemnitatibus et festis B.M.V.,Ordinarium missae,Anonymous
3,CRIM_Model_0004,Missa IX (Cum iubilo) - Gloria,Missa IX (Cum iubilo) - Gloria,[https://crimproject.org/pdf/CRIM_Model_0004.pdf],[https://crimproject.org/mei/CRIM_Model_0004.mei],10xx,1000.0,1,In solemnitatibus et festis B.M.V.,Ordinarium missae,Anonymous
4,CRIM_Model_0005,Credo I,Credo I,[https://crimproject.org/pdf/CRIM_Model_0005.pdf],[https://crimproject.org/mei/CRIM_Model_0005.mei],10xx,1000.0,1,,Ordinarium missae,Anonymous


### Inspect the DataFrame

A few essential methods for getting to know a new DataFrame:

| Method / Attribute | What it shows |
|--------------------|---------------|
| `df.head(n)` | First `n` rows (default 5) |
| `df.tail(n)` | Last `n` rows (default 5) |
| `df.info()` | Column names, non-null counts, data types |
| `df.shape` | `(rows, columns)` — note: no parentheses |
| `df.describe()` | Basic statistics for numeric columns |

In [3]:
crim.head(10)

,piece_id,title,full_title,pdf_links,mei_links,date,date_sort,number_of_voices,remarks,genre,composer
0,CRIM_Model_0001,Vidi speciosam,Vidi speciosam,[https://crimproject.org/pdf/CRIM_Model_0001.pdf],[https://crimproject.org/mei/CRIM_Model_0001.mei],1538,1538.0,5,,Motet,Johannes Lupi
1,CRIM_Model_0002,O gente brunette,O gente brunette,[https://crimproject.org/pdf/CRIM_Model_0002.pdf],[https://crimproject.org/mei/CRIM_Model_0002.mei],1548,1548.0,4,,Chanson,Thomas Champion
2,CRIM_Model_0003,Missa IX (Cum iubilo) - Kyrie,Missa IX (Cum iubilo) - Kyrie,[https://crimproject.org/pdf/CRIM_Model_0003.pdf],[https://crimproject.org/mei/CRIM_Model_0003.mei],11xx,1100.0,1,In solemnitatibus et festis B.M.V.,Ordinarium missae,Anonymous
3,CRIM_Model_0004,Missa IX (Cum iubilo) - Gloria,Missa IX (Cum iubilo) - Gloria,[https://crimproject.org/pdf/CRIM_Model_0004.pdf],[https://crimproject.org/mei/CRIM_Model_0004.mei],10xx,1000.0,1,In solemnitatibus et festis B.M.V.,Ordinarium missae,Anonymous
4,CRIM_Model_0005,Credo I,Credo I,[https://crimproject.org/pdf/CRIM_Model_0005.pdf],[https://crimproject.org/mei/CRIM_Model_0005.mei],10xx,1000.0,1,,Ordinarium missae,Anonymous
5,CRIM_Model_0006,Missa IX (Cum iubilo) - Sanctus,Missa IX (Cum iubilo) - Sanctus,[https://crimproject.org/pdf/CRIM_Model_0006.pdf],[https://crimproject.org/mei/CRIM_Model_0006.mei],13xx,1300.0,1,In solemnitatibus et festis B.M.V.,Ordinarium missae,Anonymous
6,CRIM_Model_0007,Missa IX (Cum iubilo) - Agnus Dei,Missa IX (Cum iubilo) - Agnus Dei,[https://crimproject.org/pdf/CRIM_Model_0007.pdf],[https://crimproject.org/mei/CRIM_Model_0007.mei],12xx,1200.0,1,In solemnitatibus et festis B.M.V.,Ordinarium missae,Anonymous
7,CRIM_Model_0008,Ave Maria,Ave Maria,[https://crimproject.org/pdf/CRIM_Model_0008.pdf],[https://crimproject.org/mei/CRIM_Model_0008.mei],1502,1502.0,4,,Motet,Josquin Des Prez
8,CRIM_Model_0009,Je suis déshéritée,Je suis déshéritée,[https://crimproject.org/pdf/CRIM_Model_0009.pdf],[https://crimproject.org/mei/CRIM_Model_0009.mei],1540,1540.0,4,,Chanson,Pierre Cadéac
9,CRIM_Model_0010,Quare fremuerunt gentes,Quare fremuerunt gentes,[https://crimproject.org/pdf/CRIM_Model_0010.pdf],[https://crimproject.org/mei/CRIM_Model_0010.mei],1542,1542.0,5,,Motet,Claudin de Sermisy


In [4]:
crim.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   piece_id          52 non-null     object 
 1   title             52 non-null     object 
 2   full_title        52 non-null     object 
 3   pdf_links         52 non-null     object 
 4   mei_links         52 non-null     object 
 5   date              52 non-null     object 
 6   date_sort         47 non-null     float64
 7   number_of_voices  52 non-null     int64  
 8   remarks           52 non-null     object 
 9   genre             52 non-null     object 
 10  composer          52 non-null     object 
dtypes: float64(1), int64(1), object(9)
memory usage: 4.6+ KB


In [5]:
crim.shape

(52, 11)

In [6]:
crim.describe()

,date_sort,number_of_voices
count,47.000000,52.000000
mean,1495.659574,3.846154
std,142.239277,1.661379
min,1000.000000,0.000000
25%,1529.500000,4.000000
50%,1539.000000,4.000000
75%,1555.000000,5.000000
max,1585.000000,6.000000


## 2. Working with Rows

By default Pandas shows only the first and last five rows. Here are your options:

| Method | What it does |
|--------|-------------|
| `df.head(n)` | First `n` rows |
| `df.tail(n)` | Last `n` rows |
| `df.sample(n)` | Random sample of `n` rows |
| `pd.set_option('display.max_rows', None)` | Show all rows |

In [7]:
crim.head(5)
# crim.tail(5)
# crim.sample(10)

,piece_id,title,full_title,pdf_links,mei_links,date,date_sort,number_of_voices,remarks,genre,composer
0,CRIM_Model_0001,Vidi speciosam,Vidi speciosam,[https://crimproject.org/pdf/CRIM_Model_0001.pdf],[https://crimproject.org/mei/CRIM_Model_0001.mei],1538,1538.0,5,,Motet,Johannes Lupi
1,CRIM_Model_0002,O gente brunette,O gente brunette,[https://crimproject.org/pdf/CRIM_Model_0002.pdf],[https://crimproject.org/mei/CRIM_Model_0002.mei],1548,1548.0,4,,Chanson,Thomas Champion
2,CRIM_Model_0003,Missa IX (Cum iubilo) - Kyrie,Missa IX (Cum iubilo) - Kyrie,[https://crimproject.org/pdf/CRIM_Model_0003.pdf],[https://crimproject.org/mei/CRIM_Model_0003.mei],11xx,1100.0,1,In solemnitatibus et festis B.M.V.,Ordinarium missae,Anonymous
3,CRIM_Model_0004,Missa IX (Cum iubilo) - Gloria,Missa IX (Cum iubilo) - Gloria,[https://crimproject.org/pdf/CRIM_Model_0004.pdf],[https://crimproject.org/mei/CRIM_Model_0004.mei],10xx,1000.0,1,In solemnitatibus et festis B.M.V.,Ordinarium missae,Anonymous
4,CRIM_Model_0005,Credo I,Credo I,[https://crimproject.org/pdf/CRIM_Model_0005.pdf],[https://crimproject.org/mei/CRIM_Model_0005.mei],10xx,1000.0,1,,Ordinarium missae,Anonymous


### Selecting Rows: `iloc` and `loc`

#### `iloc` — select by **integer index position**

Syntax: `df.iloc[start_row : end_row, start_col : end_col]`

- The range is *inclusive* at the start and *exclusive* at the end: `iloc[0:5]` gives rows 0–4.
- Omit either end to go from the beginning or to the end: `iloc[:5]` = first 5 rows.
- Omit the column range to get all columns.
- Negative indices count from the end: `-1` is the last column.

#### `loc` — select by **label**

Most useful when selecting columns by name, or when the index is a string.

In [8]:
# Rows 5–9, all columns
crim.iloc[5:10]

,piece_id,title,full_title,pdf_links,mei_links,date,date_sort,number_of_voices,remarks,genre,composer
5,CRIM_Model_0006,Missa IX (Cum iubilo) - Sanctus,Missa IX (Cum iubilo) - Sanctus,[https://crimproject.org/pdf/CRIM_Model_0006.pdf],[https://crimproject.org/mei/CRIM_Model_0006.mei],13xx,1300.0,1,In solemnitatibus et festis B.M.V.,Ordinarium missae,Anonymous
6,CRIM_Model_0007,Missa IX (Cum iubilo) - Agnus Dei,Missa IX (Cum iubilo) - Agnus Dei,[https://crimproject.org/pdf/CRIM_Model_0007.pdf],[https://crimproject.org/mei/CRIM_Model_0007.mei],12xx,1200.0,1,In solemnitatibus et festis B.M.V.,Ordinarium missae,Anonymous
7,CRIM_Model_0008,Ave Maria,Ave Maria,[https://crimproject.org/pdf/CRIM_Model_0008.pdf],[https://crimproject.org/mei/CRIM_Model_0008.mei],1502,1502.0,4,,Motet,Josquin Des Prez
8,CRIM_Model_0009,Je suis déshéritée,Je suis déshéritée,[https://crimproject.org/pdf/CRIM_Model_0009.pdf],[https://crimproject.org/mei/CRIM_Model_0009.mei],1540,1540.0,4,,Chanson,Pierre Cadéac
9,CRIM_Model_0010,Quare fremuerunt gentes,Quare fremuerunt gentes,[https://crimproject.org/pdf/CRIM_Model_0010.pdf],[https://crimproject.org/mei/CRIM_Model_0010.mei],1542,1542.0,5,,Motet,Claudin de Sermisy


In [9]:
# All rows, first 4 columns
crim.iloc[:, 0:4]

,piece_id,title,full_title,pdf_links
0,CRIM_Model_0001,Vidi speciosam,Vidi speciosam,[https://crimproject.org/pdf/CRIM_Model_0001.pdf]
1,CRIM_Model_0002,O gente brunette,O gente brunette,[https://crimproject.org/pdf/CRIM_Model_0002.pdf]
2,CRIM_Model_0003,Missa IX (Cum iubilo) - Kyrie,Missa IX (Cum iubilo) - Kyrie,[https://crimproject.org/pdf/CRIM_Model_0003.pdf]
3,CRIM_Model_0004,Missa IX (Cum iubilo) - Gloria,Missa IX (Cum iubilo) - Gloria,[https://crimproject.org/pdf/CRIM_Model_0004.pdf]
4,CRIM_Model_0005,Credo I,Credo I,[https://crimproject.org/pdf/CRIM_Model_0005.pdf]
5,CRIM_Model_0006,Missa IX (Cum iubilo) - Sanctus,Missa IX (Cum iubilo) - Sanctus,[https://crimproject.org/pdf/CRIM_Model_0006.pdf]
6,CRIM_Model_0007,Missa IX (Cum iubilo) - Agnus Dei,Missa IX (Cum iubilo) - Agnus Dei,[https://crimproject.org/pdf/CRIM_Model_0007.pdf]
7,CRIM_Model_0008,Ave Maria,Ave Maria,[https://crimproject.org/pdf/CRIM_Model_0008.pdf]
8,CRIM_Model_0009,Je suis déshéritée,Je suis déshéritée,[https://crimproject.org/pdf/CRIM_Model_0009.pdf]
9,CRIM_Model_0010,Quare fremuerunt gentes,Quare fremuerunt gentes,[https://crimproject.org/pdf/CRIM_Model_0010.pdf]


In [10]:
# Rows 0–4, last column only
crim.iloc[0:5, -1]

0      Johannes Lupi
1    Thomas Champion
2          Anonymous
3          Anonymous
4          Anonymous
Name: composer, dtype: object

### Dropping Rows

Remove rows by index number with `.drop()`, then restore a clean 0-based index with `.reset_index(drop=True)`.

In [11]:
crim_trimmed = crim.drop([0, 1])
crim_trimmed = crim_trimmed.reset_index(drop=True)
crim_trimmed.head(5)

,piece_id,title,full_title,pdf_links,mei_links,date,date_sort,number_of_voices,remarks,genre,composer
0,CRIM_Model_0003,Missa IX (Cum iubilo) - Kyrie,Missa IX (Cum iubilo) - Kyrie,[https://crimproject.org/pdf/CRIM_Model_0003.pdf],[https://crimproject.org/mei/CRIM_Model_0003.mei],11xx,1100.0,1,In solemnitatibus et festis B.M.V.,Ordinarium missae,Anonymous
1,CRIM_Model_0004,Missa IX (Cum iubilo) - Gloria,Missa IX (Cum iubilo) - Gloria,[https://crimproject.org/pdf/CRIM_Model_0004.pdf],[https://crimproject.org/mei/CRIM_Model_0004.mei],10xx,1000.0,1,In solemnitatibus et festis B.M.V.,Ordinarium missae,Anonymous
2,CRIM_Model_0005,Credo I,Credo I,[https://crimproject.org/pdf/CRIM_Model_0005.pdf],[https://crimproject.org/mei/CRIM_Model_0005.mei],10xx,1000.0,1,,Ordinarium missae,Anonymous
3,CRIM_Model_0006,Missa IX (Cum iubilo) - Sanctus,Missa IX (Cum iubilo) - Sanctus,[https://crimproject.org/pdf/CRIM_Model_0006.pdf],[https://crimproject.org/mei/CRIM_Model_0006.mei],13xx,1300.0,1,In solemnitatibus et festis B.M.V.,Ordinarium missae,Anonymous
4,CRIM_Model_0007,Missa IX (Cum iubilo) - Agnus Dei,Missa IX (Cum iubilo) - Agnus Dei,[https://crimproject.org/pdf/CRIM_Model_0007.pdf],[https://crimproject.org/mei/CRIM_Model_0007.mei],12xx,1200.0,1,In solemnitatibus et festis B.M.V.,Ordinarium missae,Anonymous


## 3. Working with Columns

| # | Task |
|---|------|
| 1 | Show all column names |
| 2 | Add a column |
| 3 | Drop a column |
| 4 | Rename a column (or columns) |
| 5 | Show column data types |
| 6 | Reorder or subset columns |

### Show All Column Names

Note the absence of `()` — `.columns` is an attribute, not a method.

In [12]:
# As an Index object
crim.columns

Index(['piece_id', 'title', 'full_title', 'pdf_links', 'mei_links', 'date',
       'date_sort', 'number_of_voices', 'remarks', 'genre', 'composer'],
      dtype='object')

In [13]:
# Sorted alphabetically
crim.columns.sort_values()

Index(['composer', 'date', 'date_sort', 'full_title', 'genre', 'mei_links',
       'number_of_voices', 'pdf_links', 'piece_id', 'remarks', 'title'],
      dtype='object')

### Add a Column

Assign a new column name to an expression evaluated row-by-row.  
Here we flag pieces with more than 4 voices as "large ensemble".

In [14]:
crim['large_ensemble'] = crim['number_of_voices'] > 4
crim[['title', 'number_of_voices', 'large_ensemble']].head(10)

,title,number_of_voices,large_ensemble
0,Vidi speciosam,5,True
1,O gente brunette,4,False
2,Missa IX (Cum iubilo) - Kyrie,1,False
3,Missa IX (Cum iubilo) - Gloria,1,False
4,Credo I,1,False
5,Missa IX (Cum iubilo) - Sanctus,1,False
6,Missa IX (Cum iubilo) - Agnus Dei,1,False
7,Ave Maria,4,False
8,Je suis déshéritée,4,False
9,Quare fremuerunt gentes,5,True


### Drop a Column

Columns to drop must be passed as a **list**, even if there is only one.

In [15]:
crim = crim.drop(columns=['large_ensemble'])
crim.columns

Index(['piece_id', 'title', 'full_title', 'pdf_links', 'mei_links', 'date',
       'date_sort', 'number_of_voices', 'remarks', 'genre', 'composer'],
      dtype='object')

### Rename Columns

**Option 1** — copy-then-drop (useful for a single column):

In [16]:
# Rename 'full_title' to 'long_title'
crim['long_title'] = crim['full_title']
crim = crim.drop(columns=['full_title'])
crim.columns

Index(['piece_id', 'title', 'pdf_links', 'mei_links', 'date', 'date_sort',
       'number_of_voices', 'remarks', 'genre', 'composer', 'long_title'],
      dtype='object')

**Option 2** — rename via a dictionary (cleaner for multiple columns at once):

In [17]:
col_dict = {
    'long_title':       'full_title',   # rename back for tidiness
    'number_of_voices': 'voices'
}

crim = crim.rename(columns=col_dict)
crim.columns

Index(['piece_id', 'title', 'pdf_links', 'mei_links', 'date', 'date_sort',
       'voices', 'remarks', 'genre', 'composer', 'full_title'],
      dtype='object')

**Tip: `dict.fromkeys()` as a shortcut**

Generate a skeleton dictionary with your current column names as keys. Fill in only the ones you want to rename — handy when there are many columns.

In [18]:
col_dict = dict.fromkeys(crim.columns)
col_dict

{'piece_id': None,
 'title': None,
 'pdf_links': None,
 'mei_links': None,
 'date': None,
 'date_sort': None,
 'voices': None,
 'remarks': None,
 'genre': None,
 'composer': None,
 'full_title': None}

### Show Column Data Types

Each column has a **dtype** controlling which operations are valid.  
You cannot apply string methods to a numeric column, or math to a string column.

In [19]:
crim.dtypes

piece_id       object
title          object
pdf_links      object
mei_links      object
date           object
date_sort     float64
voices          int64
remarks        object
genre          object
composer       object
full_title     object
dtype: object

### Reorder Columns / Create a Subset

Pass a list of column names in the desired order. Omit any column to exclude it — this is how you create a **subset DataFrame**.

In [20]:
column_list = ['piece_id', 'composer', 'title', 'genre', 'date_sort', 'voices']
crim_short = crim[column_list]
crim_short.head(10)

,piece_id,composer,title,genre,date_sort,voices
0,CRIM_Model_0001,Johannes Lupi,Vidi speciosam,Motet,1538.0,5
1,CRIM_Model_0002,Thomas Champion,O gente brunette,Chanson,1548.0,4
2,CRIM_Model_0003,Anonymous,Missa IX (Cum iubilo) - Kyrie,Ordinarium missae,1100.0,1
3,CRIM_Model_0004,Anonymous,Missa IX (Cum iubilo) - Gloria,Ordinarium missae,1000.0,1
4,CRIM_Model_0005,Anonymous,Credo I,Ordinarium missae,1000.0,1
5,CRIM_Model_0006,Anonymous,Missa IX (Cum iubilo) - Sanctus,Ordinarium missae,1300.0,1
6,CRIM_Model_0007,Anonymous,Missa IX (Cum iubilo) - Agnus Dei,Ordinarium missae,1200.0,1
7,CRIM_Model_0008,Josquin Des Prez,Ave Maria,Motet,1502.0,4
8,CRIM_Model_0009,Pierre Cadéac,Je suis déshéritée,Chanson,1540.0,4
9,CRIM_Model_0010,Claudin de Sermisy,Quare fremuerunt gentes,Motet,1542.0,5


### A Column is a Series

An individual column is a **Series** — a labelled one-dimensional array with its own methods.

| Expression | What it returns |
|------------|----------------|
| `df["col"]` | The column as a Series |
| `df["col"].unique()` | Array of unique values |
| `df["col"].nunique()` | Count of unique values |
| `df["col"].value_counts()` | Frequency of each value |

In [21]:
print("Genres:", crim["genre"].unique())
print("\nUnique composers:", crim["composer"].nunique())

Genres: ['Motet' 'Chanson' 'Ordinarium missae' 'Plainchant' 'Madrigal']

Unique composers: 31


## 4. Sort and Count

Pandas has built-in methods for sorting and summarizing data — no loops needed.

### Sort Values

`sort_values()` sorts by any column, ascending by default.

In [22]:
# Earliest pieces first
crim.sort_values("date_sort").head(10)

,piece_id,title,pdf_links,mei_links,date,date_sort,voices,remarks,genre,composer,full_title
3,CRIM_Model_0004,Missa IX (Cum iubilo) - Gloria,[https://crimproject.org/pdf/CRIM_Model_0004.pdf],[https://crimproject.org/mei/CRIM_Model_0004.mei],10xx,1000.0,1,In solemnitatibus et festis B.M.V.,Ordinarium missae,Anonymous,Missa IX (Cum iubilo) - Gloria
4,CRIM_Model_0005,Credo I,[https://crimproject.org/pdf/CRIM_Model_0005.pdf],[https://crimproject.org/mei/CRIM_Model_0005.mei],10xx,1000.0,1,,Ordinarium missae,Anonymous,Credo I
2,CRIM_Model_0003,Missa IX (Cum iubilo) - Kyrie,[https://crimproject.org/pdf/CRIM_Model_0003.pdf],[https://crimproject.org/mei/CRIM_Model_0003.mei],11xx,1100.0,1,In solemnitatibus et festis B.M.V.,Ordinarium missae,Anonymous,Missa IX (Cum iubilo) - Kyrie
6,CRIM_Model_0007,Missa IX (Cum iubilo) - Agnus Dei,[https://crimproject.org/pdf/CRIM_Model_0007.pdf],[https://crimproject.org/mei/CRIM_Model_0007.mei],12xx,1200.0,1,In solemnitatibus et festis B.M.V.,Ordinarium missae,Anonymous,Missa IX (Cum iubilo) - Agnus Dei
5,CRIM_Model_0006,Missa IX (Cum iubilo) - Sanctus,[https://crimproject.org/pdf/CRIM_Model_0006.pdf],[https://crimproject.org/mei/CRIM_Model_0006.mei],13xx,1300.0,1,In solemnitatibus et festis B.M.V.,Ordinarium missae,Anonymous,Missa IX (Cum iubilo) - Sanctus
21,CRIM_Model_0022,Benedicta es,[https://crimproject.org/pdf/CRIM_Model_0022.pdf],[https://crimproject.org/mei/CRIM_Model_0022.mei],13xx,1300.0,1,Authorship attributed.,Plainchant,Notker Balbulus,Benedicta es
7,CRIM_Model_0008,Ave Maria,[https://crimproject.org/pdf/CRIM_Model_0008.pdf],[https://crimproject.org/mei/CRIM_Model_0008.mei],1502,1502.0,4,,Motet,Josquin Des Prez,Ave Maria
20,CRIM_Model_0021,Benedicta es,[https://crimproject.org/pdf/CRIM_Model_0021.pdf],[https://crimproject.org/mei/CRIM_Model_0021.mei],1514,1514.0,4,,Motet,Jean Mouton,Benedicta es
42,CRIM_Model_0043,Praeter rerum,[https://crimproject.org/pdf/CRIM_Model_0043.pdf],[https://crimproject.org/mei/CRIM_Model_0043.mei],before 1521,1521.0,6,,Motet,Josquin Des Prez,Praeter rerum
29,CRIM_Model_0030,Tant que vivray,[https://crimproject.org/pdf/CRIM_Model_0030.pdf],[https://crimproject.org/mei/CRIM_Model_0030.mei],before 1528,1528.0,4,,Chanson,Claudin de Sermisy,Tant que vivray


In [23]:
# Most voices first
crim.sort_values("voices", ascending=False).head(10)

,piece_id,title,pdf_links,mei_links,date,date_sort,voices,remarks,genre,composer,full_title
16,CRIM_Model_0017,Benedicta es,[https://crimproject.org/pdf/CRIM_Model_0017.pdf],[https://crimproject.org/mei/CRIM_Model_0017.mei],1537,1537.0,6,,Motet,Josquin Des Prez,Benedicta es
49,CRIM_Model_0050,Domine Dominus noster,[https://crimproject.org/pdf/CRIM_Model_0050.pdf],[https://crimproject.org/mei/CRIM_Model_0050.mei],before 1578,1578.0,6,,Motet,Roland de Lassus,Domine Dominus noster
43,CRIM_Model_0044,Nasce la gioia mia,[https://crimproject.org/pdf/CRIM_Model_0044.pdf],[https://crimproject.org/mei/CRIM_Model_0044.mei],before 1566,1566.0,6,,Madrigal,Giovan Leonardo Primavera,Nasce la gioia mia
42,CRIM_Model_0043,Praeter rerum,[https://crimproject.org/pdf/CRIM_Model_0043.pdf],[https://crimproject.org/mei/CRIM_Model_0043.mei],before 1521,1521.0,6,,Motet,Josquin Des Prez,Praeter rerum
37,CRIM_Model_0038,Ultimi miei sospiri,[https://crimproject.org/pdf/CRIM_Model_0038.pdf],[https://crimproject.org/mei/CRIM_Model_0038.mei],before 1546,1546.0,6,,Madrigal,"Verdelot, Philippe",Ultimi miei sospiri
22,CRIM_Model_0023,Benedicta es,[https://crimproject.org/pdf/CRIM_Model_0023.pdf],[https://crimproject.org/mei/CRIM_Model_0023.mei],1539,1539.0,6,,Motet,Loyset Piéton,Benedicta es
17,CRIM_Model_0018,Baisez moy,[https://crimproject.org/pdf/CRIM_Model_0018.pdf],[https://crimproject.org/mei/CRIM_Model_0018.mei],1545,1545.0,6,,Chanson,Josquin Des Prez,Baisez moy
0,CRIM_Model_0001,Vidi speciosam,[https://crimproject.org/pdf/CRIM_Model_0001.pdf],[https://crimproject.org/mei/CRIM_Model_0001.mei],1538,1538.0,5,,Motet,Johannes Lupi,Vidi speciosam
47,CRIM_Model_0048,Sicut lilium,[https://crimproject.org/pdf/CRIM_Model_0048.pdf],[https://crimproject.org/mei/CRIM_Model_0048.mei],before 1585,1585.0,5,,Motet,Giovanni Pierluigi da Palestrina,Sicut lilium
46,CRIM_Model_0047,Nigra sum,[https://crimproject.org/pdf/CRIM_Model_0047.pdf],[https://crimproject.org/mei/CRIM_Model_0047.mei],before 1533,NaN,5,,Motet,Jean Lhéritier,Nigra sum


### Count Values

`value_counts()` returns the frequency of each unique value in a column.

In [24]:
crim['genre'].value_counts()

genre
Motet                27
Chanson              11
Madrigal              6
Ordinarium missae     5
Plainchant            3
Name: count, dtype: int64

In [25]:
crim['composer'].value_counts()

composer
Anonymous                           6
Josquin Des Prez                    5
Claudin de Sermisy                  5
Cipriano de Rore                    3
Roland de Lassus                    2
Verdelot, Philippe                  2
Notker Balbulus                     2
Giovanni Pierluigi da Palestrina    2
Pierre Sandrin                      2
Victoria, Tomás Luis de             2
Domenico Ferrabosco                 1
Cristóbal de Morales                1
Jean Lhéritier                      1
Giovan Leonardo Primavera           1
Richafort, Johannes                 1
Mantua, Jachet de                   1
Courtois, Jean                      1
Jacob Clemens non Papa              1
Padovano, Annibale                  1
Johannes Lupi                       1
Lupus Hellinck                      1
Nicolas Gombert                     1
Le Heurteur, Guillaume              1
Thomas Champion                     1
Didier Lupi                         1
Loyset Piéton                       1
Jea

In [26]:
# Store result as a new DataFrame
genre_counts = pd.DataFrame(crim['genre'].value_counts())
genre_counts

,count
genre,
Motet,27
Chanson,11
Madrigal,6
Ordinarium missae,5
Plainchant,3


## 5. Combining DataFrames

Even when working with a single source, you often split a DataFrame into subsets and then reassemble them. The two main tools are:

| Operation | When to use |
|-----------|-------------|
| **`pd.concat()`** | Stack DataFrames that share the same columns (add more rows) |
| **`pd.merge()`** | Join two DataFrames on a shared column (add more columns) |

### Concatenation

Here we split `crim` into two genre-based subsets, then stack them back into one frame.

In [27]:
motets   = crim[crim['genre'] == 'motet']
chansons = crim[crim['genre'] == 'chanson']

print(f"Motets: {len(motets)}  |  Chansons: {len(chansons)}")

combined = pd.concat([motets, chansons]).reset_index(drop=True)
print(f"Combined: {len(combined)}")
combined[['piece_id', 'composer', 'title', 'genre']]

Motets: 0  |  Chansons: 0
Combined: 0


,piece_id,composer,title,genre


### Merging

`pd.merge()` joins two DataFrames that share a common column.

Key arguments:
- `left_on` / `right_on` — which column from each frame to match on
- `how="inner"` — keep only rows that have a match in **both** frames

Here we build a small lookup table mapping composers to their birth centuries, then merge it into our main DataFrame.

In [28]:
# A small reference table
century_lookup = pd.DataFrame([
    {'composer': 'Lupi, Johannes',   'century': '16th'},
    {'composer': 'Champion, Thomas', 'century': '16th'},
    {'composer': 'Gombert, Nicolas', 'century': '16th'},
    {'composer': 'Josquin des Prez', 'century': '15th-16th'},
])

crim_merged = pd.merge(
    left=crim_short,
    right=century_lookup,
    on='composer',
    how='left'        # keep all CRIM rows; NaN where no match
)

crim_merged.head(10)

,piece_id,composer,title,genre,date_sort,voices,century
0,CRIM_Model_0001,Johannes Lupi,Vidi speciosam,Motet,1538.0,5,NaN
1,CRIM_Model_0002,Thomas Champion,O gente brunette,Chanson,1548.0,4,NaN
2,CRIM_Model_0003,Anonymous,Missa IX (Cum iubilo) - Kyrie,Ordinarium missae,1100.0,1,NaN
3,CRIM_Model_0004,Anonymous,Missa IX (Cum iubilo) - Gloria,Ordinarium missae,1000.0,1,NaN
4,CRIM_Model_0005,Anonymous,Credo I,Ordinarium missae,1000.0,1,NaN
5,CRIM_Model_0006,Anonymous,Missa IX (Cum iubilo) - Sanctus,Ordinarium missae,1300.0,1,NaN
6,CRIM_Model_0007,Anonymous,Missa IX (Cum iubilo) - Agnus Dei,Ordinarium missae,1200.0,1,NaN
7,CRIM_Model_0008,Josquin Des Prez,Ave Maria,Motet,1502.0,4,NaN
8,CRIM_Model_0009,Pierre Cadéac,Je suis déshéritée,Chanson,1540.0,4,NaN
9,CRIM_Model_0010,Claudin de Sermisy,Quare fremuerunt gentes,Motet,1542.0,5,NaN


### Cleaning Before Merging

A merge that returns fewer matches than expected usually means the key column is not formatted consistently between frames. Normalising to lowercase before merging is a simple first fix:

In [29]:
crim_lower   = crim_short.copy()
lookup_lower = century_lookup.copy()

crim_lower['composer']   = crim_lower['composer'].str.lower()
lookup_lower['composer'] = lookup_lower['composer'].str.lower()

crim_merged_clean = pd.merge(
    left=crim_lower,
    right=lookup_lower,
    on='composer',
    how='left'
)

crim_merged_clean.dropna(subset=['century']).head(10)

,piece_id,composer,title,genre,date_sort,voices,century
7,CRIM_Model_0008,josquin des prez,Ave Maria,Motet,1502.0,4,15th-16th
15,CRIM_Model_0016,josquin des prez,Mente Tota (from Vultuum tuum),Motet,1531.0,4,15th-16th
16,CRIM_Model_0017,josquin des prez,Benedicta es,Motet,1537.0,6,15th-16th
17,CRIM_Model_0018,josquin des prez,Baisez moy,Chanson,1545.0,6,15th-16th
42,CRIM_Model_0043,josquin des prez,Praeter rerum,Motet,1521.0,6,15th-16th


## 6. Making the Most of Pandas

Pandas is designed so that you almost **never need to write a `for` loop** — there is a built-in method for nearly every common operation on columns and DataFrames.

Before writing custom code, search the [documentation](https://pandas.pydata.org/about/) or the [cheat sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf). The built-in path is almost always faster, more readable, and less error-prone.

**Key principle**: clean your data *before* analysing it. The lowercase-before-merge example above illustrates why — a small normalisation step can dramatically change your results.

Continue to: [Pandas: Clean Data](https://github.com/RichardFreedman/Encoding_Music/blob/main/01_Tutorials/05_Pandas_Clean_Data.md)